# Week 61, Day 2: Context manager classes

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w061d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Temporarily:
    def __init__(self, settings, key, value):
        self.settings, self.key, self.value = settings, key, value

    def __enter__(self):
        self.old = self.settings[self.key]
        self.settings[self.key] = self.value
        return self.settings

    def __exit__(self, exc_type, exc, tb):
        self.settings[self.key] = self.old

settings = {"colour": "blue", "size": 10}
with Temporarily(settings, "colour", "red") as s:
    print(s["colour"])
print(settings["colour"])

In [ ]:
import time

class Timer:
    def __init__(self, clock=time.perf_counter):
        self.clock = clock
        self.elapsed = None

    def __enter__(self):
        self.start = self.clock()
        return self

    def __exit__(self, exc_type, exc, tb):
        self.elapsed = self.clock() - self.start

with Timer() as t:
    sum(range(100_000))
print(t.elapsed < 1)

ticks = iter([10.0, 12.5])
with Timer(clock=lambda: next(ticks)) as fake:
    pass
print(fake.elapsed)

In [ ]:
_MISSING = object()

class Patched:
    """Set obj.name = value for the block, then put things back exactly."""
    def __init__(self, obj, name, value):
        self.obj, self.name, self.value = obj, name, value

    def __enter__(self):
        self.old = getattr(self.obj, self.name, _MISSING)
        setattr(self.obj, self.name, self.value)
        return self.obj

    def __exit__(self, *exc):
        if self.old is _MISSING:
            delattr(self.obj, self.name)
        else:
            setattr(self.obj, self.name, self.old)

class Config:
    debug = False

with Patched(Config, "debug", True):
    print(Config.debug)
with Patched(Config, "verbose", 3):
    print(Config.verbose)
print(Config.debug, hasattr(Config, "verbose"))

In [ ]:
class Ignoring:
    def __init__(self, *types):
        self.types = types
        self.caught = None

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        if exc_type is not None and issubclass(exc_type, self.types):
            self.caught = exc
            return True                    # suppress just these
        return False

with Ignoring(KeyError) as ig:
    {}["missing"]
print("carried on; caught:", repr(ig.caught))

try:
    with Ignoring(KeyError):
        1 / 0
except ZeroDivisionError:
    print("other errors still escape")

In [ ]:
import copy

class Transaction:
    def __init__(self, data):
        self.data = data

    def __enter__(self):
        self.snapshot = copy.deepcopy(self.data)
        return self.data

    def __exit__(self, exc_type, exc, tb):
        if exc_type is not None:
            self.data.clear()
            self.data.update(self.snapshot)      # roll back, in place
        return False

accounts = {"ada": 100, "ben": 20}
try:
    with Transaction(accounts) as acc:
        acc["ada"] -= 50
        acc["ben"] += 50
        raise RuntimeError("the bank's computer crashed")
except RuntimeError:
    pass
print(accounts)

In [ ]:
class Indented:
    def __init__(self):
        self.depth = 0
        self.lines = []

    def __enter__(self):
        self.depth += 1
        return self

    def __exit__(self, *exc):
        self.depth -= 1

    def say(self, text):
        self.lines.append("  " * self.depth + text)

out = Indented()
out.say("start")
with out:
    out.say("level one")
    with out:
        out.say("level two")
    out.say("back to one")
out.say("end")
print("\n".join(out.lines))

> **Key idea.** Write a context manager class with __enter__ (set up, and return what the body should use: often self, sometimes a resource) and __exit__ (clean up, whatever happened). Take clocks and similar dependencies as parameters so tests are exact. For temporary changes, save in __enter__ and restore in __exit__ unconditionally, deleting what didn't exist before. Use __exit__'s arguments to roll back on failure or to suppress only the exception types you expect; return False otherwise, and keep __exit__ simple because an error there replaces the original. Managers can be one-shot, reusable (fresh state on each enter) or reentrant (a depth counter, always decremented).

### Exercise: A timer with laps

Write `Stopwatch(clock=time.perf_counter)`: a reusable context manager. `__enter__` returns the stopwatch; after each `with` block, `last` is that block's duration and `laps` is the list of all durations so far; `total()` is their sum. Record the duration even if the block raises (and let the error carry on).

In [ ]:
import time

class Stopwatch:
    def __init__(self, clock=time.perf_counter):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_timer():\n    \"a reusable manager\"\n    ticks = iter([0.0, 1.5, 10.0, 10.25, 20.0, 23.0])\n    sw = Stopwatch(clock=lambda: next(ticks))\n    with sw as s:\n        assert s is sw\n    with sw:\n        pass\n    assert sw.last == 0.25 and sw.laps == [1.5, 0.25]\n    try:\n        with sw:\n            raise ValueError\n    except ValueError:\n        pass\n    assert sw.laps == [1.5, 0.25, 3.0] and sw.total() == 4.75\n    real = Stopwatch()\n    with real:\n        sum(range(1000))\n    assert 0 <= real.last < 1")

### Exercise: Changing things for a while

Write a class `Patched(mapping, **changes)`: for the block, set each key of the dict to its new value, then restore **exactly**: keys that existed get their old values back, keys that didn't exist are removed. `__enter__` returns the mapping. Restore even if the body raises. It must be reusable.

In [ ]:
_MISSING = object()

class Patched:
    def __init__(self, mapping, **changes):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_patched():\n    \"temporary changes\"\n    env = {\"HOME\": \"/home/ada\", \"LANG\": \"en\"}\n    with Patched(env, LANG=\"fr\", DEBUG=\"1\") as e:\n        assert e is env and env == {\"HOME\": \"/home/ada\", \"LANG\": \"fr\", \"DEBUG\": \"1\"}\n    assert env == {\"HOME\": \"/home/ada\", \"LANG\": \"en\"}\n    p = Patched(env, HOME=\"/tmp\")\n    try:\n        with p:\n            raise RuntimeError\n    except RuntimeError:\n        pass\n    assert env[\"HOME\"] == \"/home/ada\"\n    env[\"HOME\"] = \"/changed\"\n    with p:\n        assert env[\"HOME\"] == \"/tmp\"\n    assert env[\"HOME\"] == \"/changed\", \"the old value is saved at enter time\"\n    with Patched(env, LANG=None):\n        assert env[\"LANG\"] is None\n    assert env[\"LANG\"] == \"en\"")

### Exercise: Catching only what you expect

Write `Ignoring(*types)`: a context manager that suppresses exceptions of the given types (and their subclasses) and nothing else, storing the suppressed exception in `caught` (`None` if nothing was caught) and how many times it has suppressed one in `count`, across reuses.

In [ ]:
class Ignoring:
    def __init__(self, *types):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ignoring():\n    \"deliberate suppression\"\n    ig = Ignoring(KeyError, IndexError)\n    with ig:\n        [][3]\n    assert isinstance(ig.caught, IndexError) and ig.count == 1\n    with ig:\n        pass\n    assert ig.caught is None and ig.count == 1\n    try:\n        with ig:\n            1 / 0\n        assert False\n    except ZeroDivisionError:\n        pass\n    class Mine(KeyError):\n        pass\n    with ig:\n        raise Mine(\"sub\")\n    assert ig.count == 2 and type(ig.caught) is Mine")

### Exercise: All or nothing

Write `Transaction(data)` for a dict whose values may be lists or dicts: on entry take a deep copy; on success keep the changes and record `committed = True`; on an error restore the dict **in place** (the same object, its old contents), set `committed = False`, and let the error carry on. `__enter__` returns the data. Then `transfer(accounts, source, target, amount)`: move money inside a `Transaction`, raising `ValueError("insufficient funds")` **after** taking the money if the source would go below zero, so the rollback is what saves the data.

In [ ]:
import copy

class Transaction:
    def __init__(self, data):
        ___

def transfer(accounts, source, target, amount):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_transaction():\n    \"rollback\"\n    accounts = {\"ada\": 100, \"ben\": 20}\n    ref = accounts\n    transfer(accounts, \"ada\", \"ben\", 30)\n    assert accounts == {\"ada\": 70, \"ben\": 50}\n    try:\n        transfer(accounts, \"ben\", \"ada\", 999)\n        assert False\n    except ValueError as e:\n        assert str(e) == \"insufficient funds\"\n    assert accounts == {\"ada\": 70, \"ben\": 50} and accounts is ref\n    data = {\"items\": [1, 2], \"meta\": {\"n\": 2}}\n    t = Transaction(data)\n    try:\n        with t as d:\n            d[\"items\"].append(3)\n            d[\"meta\"][\"n\"] = 3\n            d[\"new\"] = True\n            raise KeyError\n    except KeyError:\n        pass\n    assert data == {\"items\": [1, 2], \"meta\": {\"n\": 2}} and t.committed is False\n    with t as d:\n        d[\"items\"].append(9)\n    assert data[\"items\"] == [1, 2, 9] and t.committed is True")

### Exercise: Nesting depth

Write `Section(log)`: a reentrant context manager for nested headings. Each `with` increases the depth; `section.heading(text)` appends `"#" * depth + " " + text` to the log list (at depth 0, just the text); `__exit__` always decreases the depth, even after an error. `depth` is readable.

In [ ]:
class Section:
    def __init__(self, log):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_section():\n    \"a depth counter\"\n    log = []\n    s = Section(log)\n    s.heading(\"Title\")\n    with s:\n        s.heading(\"Part\")\n        with s:\n            s.heading(\"Detail\")\n        s.heading(\"Part two\")\n    assert log == [\"Title\", \"# Part\", \"## Detail\", \"# Part two\"] and s.depth == 0\n    try:\n        with s:\n            with s:\n                raise ValueError\n    except ValueError:\n        pass\n    assert s.depth == 0")

### Bug hunt: Restoring the new value

This manager saves the old value **after** changing it, so it "restores" the new one and the setting stays changed: it prints `quiet quiet`. Save before changing so it prints `quiet loud`.

In [ ]:
settings = {"mode": "loud"}

class Quietly:
    def __enter__(self):
        settings["mode"] = "quiet"
        self.old = settings["mode"]
    def __exit__(self, *exc):
        settings["mode"] = self.old

with Quietly():
    inside = settings["mode"]
print(inside, settings["mode"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"save first\"\n    expect_output(\"quiet loud\")")

### Bug hunt: __exit__ takes four arguments

`__exit__` is always called with three extra arguments (type, value, traceback). This one accepts none, so leaving the block raises `TypeError` and it prints `TypeError`. Accept them (`*exc` is fine) so it prints `done`.

In [ ]:
class Gate:
    def __enter__(self):
        return self
    def __exit__(self):
        print("closing")

try:
    with Gate():
        pass
    print("done")
except TypeError:
    print("TypeError")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the signature\"\n    expect_output(\"closing\\ndone\")")

### Bug hunt: A depth that never comes back

This only decreases the depth when the block **succeeds**, so after an error the manager thinks it's still one level in, and the final depth printed is `1`. Decrease it whatever happened so it prints `0`.

In [ ]:
class Level:
    def __init__(self):
        self.depth = 0
    def __enter__(self):
        self.depth += 1
        return self
    def __exit__(self, exc_type, exc, tb):
        if exc_type is None:
            self.depth -= 1

lv = Level()
try:
    with lv:
        raise KeyError
except KeyError:
    pass
print(lv.depth)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"always decrement\"\n    expect_output(\"0\")")

### Puzzle: A temporary attribute

Arrange the lines of a manager that sets an attribute for the block and restores it. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the kind of manager that can be entered again while already inside: "one-shot", "reusable" or "reentrant"
q = ___   # True or False: a manager should save the old value in __init__
r = ___   # what replaces the original exception if __exit__ raises: "the new exception" or "the original"
s = ___   # issubclass(KeyError, LookupError)
t = ___   # True or False: a Transaction should restore its snapshot when the body succeeds

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"reentrant\" and q is False and r == \"the new exception\", \"p, q and r\"\n    assert s is True and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: nested savepoints

A `Ledger` keeps account balances and supports **nested** transactions, like a database's savepoints: an inner transaction that fails rolls back only its own changes, while the outer one carries on; when the outer one fails, everything since it began is undone, including inner transactions that had succeeded.

- `Ledger(balances)` holds a dict; `ledger.balance(name)`; `ledger.move(source, target, amount)` changes balances, raising `ValueError("insufficient funds")` if the source would go negative (without changing anything).
- `ledger.transaction()` returns a context manager: entering pushes a snapshot of the balances onto a stack; leaving pops it, and on an error restores that snapshot and lets the error carry on. `__enter__` returns the ledger.
- `ledger.depth()`: how many transactions are open.
- `ledger.audit`: a list of `("commit", depth)` and `("rollback", depth)` entries, recording each transaction as it ends, with the depth it had while open (outermost 1).

In [ ]:
import copy

class Ledger:
    def __init__(self, balances):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ledger():\n    \"nested savepoints\"\n    led = Ledger({\"ada\": 100, \"ben\": 0, \"cy\": 0})\n    with led.transaction():\n        led.move(\"ada\", \"ben\", 30)\n        try:\n            with led.transaction():\n                led.move(\"ben\", \"cy\", 10)\n                led.move(\"ada\", \"cy\", 500)\n        except ValueError:\n            pass\n        assert led.balance(\"cy\") == 0 and led.balance(\"ben\") == 30 and led.depth() == 1\n        with led.transaction():\n            led.move(\"ben\", \"cy\", 5)\n    assert (led.balance(\"ada\"), led.balance(\"ben\"), led.balance(\"cy\")) == (70, 25, 5) and led.depth() == 0\n    assert led.audit == [(\"rollback\", 2), (\"commit\", 2), (\"commit\", 1)]\n    try:\n        with led.transaction() as l:\n            assert l is led\n            with led.transaction():\n                led.move(\"ada\", \"cy\", 70)\n            raise RuntimeError(\"power cut\")\n    except RuntimeError:\n        pass\n    assert led.balance(\"ada\") == 70 and led.balance(\"cy\") == 5 and led.audit[-2:] == [(\"commit\", 2), (\"rollback\", 1)]\n    try:\n        led.move(\"cy\", \"ada\", 6)\n        assert False\n    except ValueError:\n        pass\n    assert led.balance(\"cy\") == 5")

## Warm-up

## Timing a block

A timer is the classic first context manager: note the time in `__enter__`, compute the difference in `__exit__`, and keep it on the object so the code after the block can read it. Returning `self` from `__enter__` makes that easy: `with Timer() as t: ...` then `t.elapsed`.

Two design points. First, take the **clock** as a parameter with a sensible default (`time.perf_counter`): tests can pass a fake clock and get exact numbers, as in Week 60's time-aware decorators. Second, record the time even when the body fails: `__exit__` runs either way, so the measurement is always there.

## Temporary changes

Many managers change something for the duration of the block and put it back afterwards: a setting, an attribute, an environment variable, the current folder, the random seed. The rules:

- **Save in `__enter__`, restore in `__exit__`.** Save inside `__enter__`, not `__init__`: the manager might be created long before it's used.
- **Restore unconditionally.** `__exit__` doesn't need to look at the error: it restores either way.
- **Handle "there was nothing there".** If the attribute or key didn't exist before, restoring means **deleting** it, not setting it to `None`. A private sentinel object (Week 59) records "missing".

## Reacting to errors

`__exit__`'s arguments let a manager behave differently on success and failure: log the error, count failures, roll back changes, or **suppress** particular errors. Suppress only what you mean: return true for the specific exception types you expect (`issubclass(exc_type, self.types)`), and `False` for everything else so real bugs still surface. Keep the caught exception on the manager if the code afterwards may want it.

If `__exit__` itself raises, that new exception replaces the original one (whose details survive as the new one's `__context__`), so keep `__exit__` simple and safe.

## All or nothing

A **transaction** applies a group of changes completely or not at all. Databases do this with commit and rollback (Phase 6). The same idea fits any data: in `__enter__`, take a **snapshot**; if the body succeeds, keep the changes; if it raises, restore the snapshot and let the error carry on. The code inside reads naturally, and partial updates can't be left behind.

For simple data a copy is a fine snapshot. (Copies have subtleties for nested data, `copy.deepcopy` versus a shallow copy: Week 62 covers them.)

## Reuse and nesting

Three kinds of manager:

- **One-shot**: can be used in only one `with`. File objects are like this (once closed, they stay closed), and so are tomorrow's generator-based managers.
- **Reusable**: can be used in several `with` statements one after another, because `__enter__` sets up fresh state each time. Today's `Timer` is reusable.
- **Reentrant**: can be entered again while already inside itself. That needs a **depth** counter: set up only when entering at depth 0, clean up only when leaving back to depth 0. `threading.RLock` (Week 66) is reentrant; an indentation helper is a friendly example.

Decrement the depth in `__exit__` **whatever happened**, or an error will leave the manager thinking it's still inside.

## Practice

## Quiz

## Daily challenge

## Before you go

Think of one thing in a program you've written that could be a context manager class (a temporary setting, a log section, a timer around slow code) and sketch its `__enter__` and `__exit__`. Tomorrow: `contextlib`, which turns a generator into a context manager in a few lines.